# NLPOLYMPIC: Transformer NMT Trung → Việt

Notebook này train một **Transformer encoder–decoder từ đầu** (không dùng mô hình pretrained, đúng luật đề) để dịch tiếng Trung giản thể sang tiếng Việt, rồi tạo file nộp cho **private test**.

So với `baseline.ipynb` (GRU không attention, greedy), notebook này có:

| Thành phần | Baseline | Notebook này |
|---|---|---|
| Model | GRU 1 lớp, hidden 128, không attention | Transformer 4+4 lớp, d_model 256, attention đầy đủ |
| Tokenizer | BPE vocab 3000 | Unigram vocab 8000, giữ nguyên định dạng `_` của tiếng Việt |
| Chia valid | 10% cuối file | 5% ngẫu nhiên theo seed |
| Regularization | dropout 0.3 | dropout 0.3 + label smoothing 0.1 (+ R-Drop tùy chọn) |
| Decode | greedy | beam search + length penalty |
| Sau train | lấy checkpoint tốt nhất | trung bình trọng số top-K checkpoint (+ ensemble nhiều seed tùy chọn) |

**Cách chạy trên Kaggle/Colab:**
1. Bật GPU (Kaggle: *Settings → Accelerator → GPU T4*; Colab: *Runtime → Change runtime type → T4 GPU*).
2. Đưa thư mục `dataset/` lên (Kaggle: *Add Input* dạng dataset; Colab: upload hoặc mount Drive). Notebook tự dò đường dẫn, nếu không thấy thì sửa `DATA_DIR`.
3. Chạy thử với `QUICK_RUN = True` (khoảng 1–2 phút) để chắc pipeline chạy, rồi đổi về `False` và *Run All*.
4. File nộp nằm trong thư mục `outputs/`: `private_submission.csv` (và `.zip`).

## 1. Cài đặt và import

In [ ]:
import importlib.util, subprocess, sys
for pkg in ["sentencepiece", "sacrebleu"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, re, math, random, time, copy, zipfile, warnings
import pandas as pd
import sentencepiece as spm
import sacrebleu
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore", message=".*enable_nested_tensor.*")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE)

## 2. Cấu hình

- `QUICK_RUN = True`: chạy thử nhanh với 2.000 câu, model nhỏ, 1 epoch. Chỉ để kiểm tra pipeline, **BLEU sẽ rất thấp**.
- `SEEDS`: để `[42]` là train 1 model. Đặt `[42, 43, 44]` để train 3 model và **ensemble** khi dịch (tốn gấp 3 thời gian, thường thêm khoảng +1 BLEU).
- `RDROP_ALPHA > 0` bật R-Drop (regularization mạnh hơn, mỗi bước chậm khoảng 2 lần). Thử `RDROP_ALPHA = 1.0` nếu model bị overfit.

In [ ]:
QUICK_RUN = False
SEEDS = [42]              # [42, 43, 44] để ensemble
RDROP_ALPHA = 0.0         # 0 = tắt R-Drop

# Đường dẫn: None = tự dò (repo local, /kaggle/input, /content)
DATA_DIR = None
OUT_DIR = "outputs"

# Tokenizer
VOCAB_ZH = 8000
VOCAB_VI = 8000

# Model
D_MODEL, N_HEAD, N_ENC, N_DEC, FFN, DROPOUT = 256, 4, 4, 4, 1024, 0.3

# Training
MAX_TOKENS = 2048         # số token (đã tính padding) mỗi batch
LR, WARMUP = 5e-4, 1000
LABEL_SMOOTHING = 0.1
EPOCHS, PATIENCE = 80, 10 # dừng sớm nếu BLEU valid không tăng sau PATIENCE epoch
TOP_K_AVG = 5             # số checkpoint tốt nhất đem trung bình
VALID_RATIO = 0.05
MAX_SRC_LEN = 128         # số subword tối đa mỗi câu

# Decode
BEAM, LENPEN = 5, 0.6

if QUICK_RUN:
    D_MODEL, N_HEAD, N_ENC, N_DEC, FFN = 128, 4, 2, 2, 256
    EPOCHS, WARMUP, TOP_K_AVG = 1, 50, 1
    VOCAB_ZH = VOCAB_VI = 2000

os.makedirs(OUT_DIR, exist_ok=True)
PAD, UNK, BOS, EOS = 0, 1, 2, 3

## 3. Đọc dữ liệu

- Đọc **mọi dòng** của file test, không bỏ dòng nào, để file nộp có đúng số dòng và đúng thứ tự như `private_test.zh`.
- Chia valid **ngẫu nhiên** 5%. Baseline lấy 10% cuối file nên tập valid không đại diện cho toàn bộ dữ liệu.
- Bỏ các cặp câu rỗng hoặc có độ dài lệch nhau bất thường (thường là cặp câu dịch sai).

In [ ]:
def find_file(name, roots=(".", "/kaggle/input", "/content"), max_depth=5):
    for root in roots:
        if not os.path.isdir(root):
            continue
        base_depth = root.rstrip("/").count("/")
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count("/") - base_depth >= max_depth:
                dirnames[:] = []
            if name in filenames:
                return os.path.join(dirpath, name)
    raise FileNotFoundError(f"Không tìm thấy {name}. Hãy đặt DATA_DIR thủ công.")

def data_path(name, sub):
    if DATA_DIR:
        return os.path.join(DATA_DIR, sub, name)
    return find_file(name)

def read_lines(path):
    with open(path, encoding="utf-8-sig") as f:
        return [" ".join(line.split()) for line in f.read().splitlines()]

TRAIN_ZH = data_path("train.zh", "train")
TRAIN_VI = data_path("train.vi", "train")
PUBLIC_ZH = data_path("public_test.zh", "public_test")
PRIVATE_ZH = data_path("private_test.zh", "private_test")
print(TRAIN_ZH, PUBLIC_ZH, PRIVATE_ZH, sep="\n")

all_zh, all_vi = read_lines(TRAIN_ZH), read_lines(TRAIN_VI)
public_zh, private_zh = read_lines(PUBLIC_ZH), read_lines(PRIVATE_ZH)
assert len(all_zh) == len(all_vi), "train.zh và train.vi lệch số dòng"

pairs = [(z, v) for z, v in zip(all_zh, all_vi) if z and v]
random.Random(0).shuffle(pairs)
n_valid = int(len(pairs) * VALID_RATIO)
valid_pairs, train_pairs = pairs[:n_valid], pairs[n_valid:]

def ok_ratio(z, v):
    a, b = len(z.split()), len(v.split())
    return max(a, b) <= 5 or max(a, b) / max(1, min(a, b)) <= 3.0

n_before = len(train_pairs)
train_pairs = [p for p in train_pairs if ok_ratio(*p)]
print(f"Bỏ {n_before - len(train_pairs)} cặp lệch độ dài")
if QUICK_RUN:
    train_pairs, valid_pairs = train_pairs[:2000], valid_pairs[:200]

print(f"train: {len(train_pairs)} | valid: {len(valid_pairs)}")
print(f"public_test: {len(public_zh)} | private_test: {len(private_zh)}")
for z, v in train_pairs[:3]:
    print(z, "=>", v)

## 4. Tokenizer SentencePiece

- Train riêng cho tiếng Trung và tiếng Việt, **chỉ trên tập train** (không nhìn valid/test).
- Dữ liệu tiếng Việt đã tách từ, từ ghép nối bằng `_` (ví dụ `thay_đổi`). Ta **giữ nguyên định dạng này** ở output vì bản dịch tham chiếu nhiều khả năng cùng định dạng với `train.vi`. Viết khác định dạng (ví dụ `thay đổi`) sẽ làm lệch n-gram và mất BLEU.
- Tiếng Việt dùng chuẩn hóa `identity` để `decode(encode(x)) == x` tuyệt đối, không làm biến đổi dấu thanh.

In [ ]:
def train_spm(lines, prefix, vocab_size, normalization):
    txt = prefix + ".txt"
    with open(txt, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    spm.SentencePieceTrainer.Train(
        input=txt, model_prefix=prefix, vocab_size=vocab_size, model_type="unigram",
        character_coverage=1.0, normalization_rule_name=normalization,
        pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS,
        hard_vocab_limit=False, num_threads=os.cpu_count(), minloglevel=2,
    )
    return spm.SentencePieceProcessor(model_file=prefix + ".model")

sp_zh = train_spm([z for z, _ in train_pairs], os.path.join(OUT_DIR, "spm_zh"), VOCAB_ZH, "nmt_nfkc")
sp_vi = train_spm([v for _, v in train_pairs], os.path.join(OUT_DIR, "spm_vi"), VOCAB_VI, "identity")
print("vocab zh:", sp_zh.get_piece_size(), "| vocab vi:", sp_vi.get_piece_size())

bad = [v for _, v in valid_pairs if sp_vi.decode(sp_vi.encode(v)) != v]
print(f"Round-trip tiếng Việt trên valid: {len(valid_pairs) - len(bad)}/{len(valid_pairs)} câu khớp")
for v in bad[:3]:
    print("  lệch:", repr(v), "->", repr(sp_vi.decode(sp_vi.encode(v))))

z, v = train_pairs[0]
print(sp_zh.encode(z, out_type=str))
print(sp_vi.encode(v, out_type=str))

## 5. Dataset và batching theo số token

Câu có độ dài rất khác nhau. Gom các câu dài xấp xỉ nhau vào một batch sẽ giảm padding, và giới hạn theo **số token** (thay vì số câu) giúp mỗi batch dùng GPU đều nhau.

In [ ]:
def encode_src(s):
    return sp_zh.encode(s)[:MAX_SRC_LEN - 1] + [EOS]

def encode_tgt(s):
    return [BOS] + sp_vi.encode(s)[:MAX_SRC_LEN - 2] + [EOS]

train_data = [(encode_src(z), encode_tgt(v)) for z, v in train_pairs]
valid_src = [encode_src(z) for z, _ in valid_pairs]
valid_ref = [v for _, v in valid_pairs]

class TokenBatchSampler:
    # Gom các câu dài gần bằng nhau, mỗi batch <= max_tokens (tính cả padding)
    def __init__(self, lengths, max_tokens, shuffle=True):
        self.lengths, self.max_tokens, self.shuffle = lengths, max_tokens, shuffle
        self.batches = self._make_batches()

    def _make_batches(self):
        noise = [random.random() if self.shuffle else 0 for _ in self.lengths]
        order = sorted(range(len(self.lengths)), key=lambda i: (self.lengths[i], noise[i]))
        batches, cur, cur_max = [], [], 0
        for i in order:
            new_max = max(cur_max, self.lengths[i])
            if cur and new_max * (len(cur) + 1) > self.max_tokens:
                batches.append(cur)
                cur, new_max = [], self.lengths[i]
            cur.append(i)
            cur_max = new_max
        if cur:
            batches.append(cur)
        if self.shuffle:
            random.shuffle(batches)
        return batches

    def __iter__(self):
        batches, self.batches = self.batches, self._make_batches()
        return iter(batches)

    def __len__(self):
        return len(self.batches)

def pad_batch(seqs):
    out = torch.full((len(seqs), max(len(s) for s in seqs)), PAD, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return out

def collate(batch):
    src, tgt = zip(*batch)
    return pad_batch(src), pad_batch(tgt)

def make_train_loader():
    lengths = [max(len(s), len(t)) for s, t in train_data]
    sampler = TokenBatchSampler(lengths, MAX_TOKENS)
    return DataLoader(train_data, batch_sampler=sampler, collate_fn=collate)

print("Số batch mỗi epoch:", len(make_train_loader()))

## 6. Model Transformer

Dùng `nn.Transformer` của PyTorch, train từ trọng số ngẫu nhiên:
- **Pre-norm** (`norm_first=True`): train ổn định hơn với model nhỏ và dữ liệu ít.
- **Positional encoding** dạng sin/cos cố định.
- **Chia sẻ trọng số** giữa embedding của decoder và lớp output: bớt tham số, đỡ overfit.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.size(1)]


class TransformerNMT(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, nhead, n_enc, n_dec, ffn, dropout):
        super().__init__()
        self.scale = math.sqrt(d_model)
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=PAD)
        for emb in (self.src_emb, self.tgt_emb):
            nn.init.normal_(emb.weight, 0, d_model ** -0.5)
            nn.init.zeros_(emb.weight[PAD])
        self.pos = PositionalEncoding(d_model)
        self.drop = nn.Dropout(dropout)
        self.transformer = nn.Transformer(
            d_model, nhead, n_enc, n_dec, ffn, dropout, batch_first=True, norm_first=True)
        self.out = nn.Linear(d_model, tgt_vocab, bias=False)
        self.out.weight = self.tgt_emb.weight

    def embed(self, emb, x):
        return self.drop(self.pos(emb(x) * self.scale))

    def encode(self, src):
        src_mask = src.eq(PAD)
        memory = self.transformer.encoder(self.embed(self.src_emb, src), src_key_padding_mask=src_mask)
        return memory, src_mask

    def decode(self, tgt_in, memory, src_mask):
        T = tgt_in.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device), 1)
        h = self.transformer.decoder(self.embed(self.tgt_emb, tgt_in), memory,
                                     tgt_mask=causal, memory_key_padding_mask=src_mask)
        return self.out(h)

    def forward(self, src, tgt_in):
        memory, src_mask = self.encode(src)
        return self.decode(tgt_in, memory, src_mask)


def build_model():
    return TransformerNMT(sp_zh.get_piece_size(), sp_vi.get_piece_size(),
                          D_MODEL, N_HEAD, N_ENC, N_DEC, FFN, DROPOUT).to(DEVICE)

print(f"Số tham số: {sum(p.numel() for p in build_model().parameters()):,}")

## 7. Beam search và đánh giá BLEU

- **Beam search** giữ `BEAM` giả thuyết tốt nhất ở mỗi bước thay vì chỉ 1 như greedy. Điểm cuối chia cho *length penalty* `((5+len)/6)^α` để không thiên vị câu ngắn.
- Hàm nhận **danh sách model**: nếu có nhiều model (ensemble), xác suất các model được lấy trung bình ở mỗi bước.
- `BEAM=1` chính là greedy.

In [ ]:
@torch.no_grad()
def beam_search(models, src, beam=BEAM, lenpen=LENPEN, max_len_a=1.5, max_len_b=10):
    B = src.size(0)
    states = []
    for m in models:
        memory, src_mask = m.encode(src)
        states.append((memory.repeat_interleave(beam, 0), src_mask.repeat_interleave(beam, 0)))
    max_len = int((~src.eq(PAD)).sum(1).max().item() * max_len_a + max_len_b)

    ys = torch.full((B * beam, 1), BOS, dtype=torch.long, device=src.device)
    scores = torch.zeros(B, beam, device=src.device)
    scores[:, 1:] = float("-inf")          # bước đầu chỉ mở rộng từ beam 0
    finished = torch.zeros(B * beam, dtype=torch.bool, device=src.device)
    base = (torch.arange(B, device=src.device) * beam).unsqueeze(1)

    for _ in range(max_len):
        logps = [m.decode(ys, mem, mask)[:, -1].float().log_softmax(-1)
                 for m, (mem, mask) in zip(models, states)]
        logp = torch.logsumexp(torch.stack(logps), 0) - math.log(len(models))
        logp[finished] = float("-inf")     # câu đã xong chỉ được nối PAD, không tốn điểm
        logp[finished, PAD] = 0.0
        V = logp.size(-1)
        top, idx = (scores.view(-1, 1) + logp).view(B, beam * V).topk(beam, -1)
        sel = (base + idx // V).view(-1)
        tok = (idx % V).view(-1, 1)
        ys = torch.cat([ys[sel], tok], 1)
        finished = finished[sel] | tok.view(-1).eq(EOS)
        scores = top
        if finished.all():
            break

    lengths = ys[:, 1:].ne(PAD).sum(1).float()
    norm = scores.view(-1) / ((5 + lengths) / 6) ** lenpen
    best = norm.view(B, beam).argmax(-1) + base.squeeze(1)
    results = []
    for row in ys[best, 1:].tolist():
        out = []
        for t in row:
            if t in (EOS, PAD):
                break
            out.append(t)
        results.append(out)
    return results


def postprocess(text):
    # Dọn "_" thừa: "__" -> "_", bỏ "_" ở đầu/cuối từ (chỉ giữ "_" nối giữa 2 âm tiết)
    text = " ".join(w.strip("_") for w in re.sub(r"_+", "_", text).split())
    # Gộp từ lặp liên tiếp >= 3 lần (lỗi thường gặp của NMT) thành 1 lần
    out, run = [], 0
    for w in text.split():
        run = run + 1 if out and out[-1] == w else 1
        if run <= 2:
            out.append(w)
        elif run == 3:
            out[-1:] = []  # đã lặp >= 3 lần: chỉ giữ 1 lần
    return " ".join(out) or "."


@torch.no_grad()
def translate_ids(models, src_ids, beam=BEAM, lenpen=LENPEN, batch_size=128):
    for m in models:
        m.eval()
    order = sorted(range(len(src_ids)), key=lambda i: len(src_ids[i]))
    hyps = [None] * len(src_ids)
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        src = pad_batch([src_ids[i] for i in idx]).to(DEVICE)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            outs = beam_search(models, src, beam, lenpen)
        for i, o in zip(idx, outs):
            hyps[i] = postprocess(sp_vi.decode(o))
    return hyps


def valid_bleu(models, beam=1, lenpen=LENPEN):
    hyps = translate_ids(models, valid_src, beam, lenpen)
    return sacrebleu.corpus_bleu(hyps, [valid_ref]).score, hyps

## 8. Huấn luyện

- **AdamW** với lịch learning rate *warmup rồi giảm theo 1/√step*, là lịch chuẩn cho Transformer.
- **Label smoothing 0.1**: không ép model tự tin 100% vào một token, giúp tổng quát tốt hơn.
- **AMP (fp16)** trên GPU để train nhanh hơn.
- Sau mỗi epoch đo **BLEU trên valid** (greedy). Giữ lại `TOP_K_AVG` checkpoint có BLEU cao nhất và dừng sớm nếu BLEU không tăng sau `PATIENCE` epoch.
- **R-Drop** (tùy chọn): chạy mỗi batch 2 lần với dropout khác nhau rồi phạt độ lệch (KL) giữa 2 phân phối.

In [ ]:
def compute_loss(model, src, tgt_in, tgt_out):
    if RDROP_ALPHA <= 0:
        logits = model(src, tgt_in)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), tgt_out.reshape(-1),
                               ignore_index=PAD, label_smoothing=LABEL_SMOOTHING)
    logits = model(torch.cat([src, src]), torch.cat([tgt_in, tgt_in])).float()
    tgt2 = torch.cat([tgt_out, tgt_out])
    ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt2.reshape(-1),
                         ignore_index=PAD, label_smoothing=LABEL_SMOOTHING)
    lp1, lp2 = logits.log_softmax(-1).chunk(2)
    mask = tgt_out.ne(PAD).unsqueeze(-1)
    kl = (F.kl_div(lp1, lp2, log_target=True, reduction="none") +
          F.kl_div(lp2, lp1, log_target=True, reduction="none"))
    kl = (kl * mask).sum() / mask.sum() / 2
    return ce + RDROP_ALPHA * kl


def average_states(states):
    avg = copy.deepcopy(states[0])
    for k in avg:
        if avg[k].is_floating_point():
            avg[k] = sum(s[k].float() for s in states) / len(states)
    return avg


def train_model(seed):
    random.seed(seed); torch.manual_seed(seed)
    model = build_model()
    loader = make_train_loader()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: min((step + 1) / WARMUP, math.sqrt(WARMUP / (step + 1))))
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

    top = []          # [(bleu, epoch, state_dict)] giữ TOP_K_AVG bản tốt nhất
    best_bleu, bad_epochs = -1.0, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        for src, tgt in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = compute_loss(model, src, tgt[:, :-1], tgt[:, 1:])
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total, n = total + loss.item(), n + 1

        bleu, _ = valid_bleu([model], beam=1)
        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        top = sorted(top + [(bleu, epoch, state)], key=lambda x: -x[0])[:TOP_K_AVG]
        mark = ""
        if bleu > best_bleu:
            best_bleu, bad_epochs, mark = bleu, 0, " *"
        else:
            bad_epochs += 1
        print(f"[seed {seed}] epoch {epoch:02d} | loss {total / n:.3f} | "
              f"lr {scheduler.get_last_lr()[0]:.2e} | valid BLEU {bleu:.2f}{mark} | {time.time() - t0:.0f}s")
        if bad_epochs >= PATIENCE:
            print(f"Dừng sớm: BLEU không tăng sau {PATIENCE} epoch.")
            break

    print(f"Top checkpoint (epoch, BLEU): {[(e, round(b, 2)) for b, e, _ in top]}")
    best_state = top[0][2]
    avg_state = average_states([s for _, _, s in top])
    torch.save(best_state, os.path.join(OUT_DIR, f"best_seed{seed}.pt"))
    torch.save(avg_state, os.path.join(OUT_DIR, f"avg_seed{seed}.pt"))
    return best_state, avg_state

In [ ]:
trained = {seed: train_model(seed) for seed in SEEDS}

## 9. Chọn cấu hình decode tốt nhất trên valid

So sánh BLEU trên valid của: checkpoint tốt nhất (greedy / beam), checkpoint trung bình (beam), và ensemble (nếu train nhiều seed). Cấu hình có BLEU cao nhất sẽ được dùng để dịch test.

In [ ]:
def load(state):
    m = build_model()
    m.load_state_dict(state)
    return m.eval()

first = SEEDS[0]
best_m, avg_m = load(trained[first][0]), load(trained[first][1])
candidates = {
    "best + greedy": ([best_m], 1),
    f"best + beam{BEAM}": ([best_m], BEAM),
    f"avg top-{TOP_K_AVG} + beam{BEAM}": ([avg_m], BEAM),
}
if len(SEEDS) > 1:
    candidates[f"ensemble {len(SEEDS)} seed (avg) + beam{BEAM}"] = ([load(trained[s][1]) for s in SEEDS], BEAM)

results = {}
for name, (models, beam) in candidates.items():
    bleu, hyps = valid_bleu(models, beam=beam)
    results[name] = (bleu, hyps)
    print(f"{name:<40} valid BLEU = {bleu:.2f}")

best_name = max(results, key=lambda k: results[k][0])
final_models, final_beam = candidates[best_name]
print(f"\n=> Dùng: {best_name}")

print("\nVí dụ dịch trên valid:")
for (z, ref), hyp in list(zip(valid_pairs, results[best_name][1]))[:10]:
    print(f"ZH : {z}\nREF: {ref}\nHYP: {hyp}\n")

### (Tùy chọn) Tinh chỉnh length penalty

BLEU phạt bản dịch ngắn (Brevity Penalty). Nếu `sacrebleu` báo `ratio` < 1 (bản dịch ngắn hơn tham chiếu), tăng `LENPEN` sẽ ra câu dài hơn. Cell này thử vài giá trị trên valid rồi giữ giá trị tốt nhất.

In [ ]:
for lp in [0.6, 1.0, 1.4]:
    bleu, hyps = valid_bleu(final_models, beam=final_beam, lenpen=lp)
    score = sacrebleu.corpus_bleu(hyps, [valid_ref])
    print(f"lenpen={lp}: BLEU {bleu:.2f} | length ratio {score.sys_len / max(1, score.ref_len):.3f}")
    if bleu > results[best_name][0]:
        results[best_name] = (bleu, hyps)
        LENPEN = lp
print("Chọn LENPEN =", LENPEN)

## 10. Dịch test và tạo file nộp

File CSV gồm 2 cột `tieng_trung`, `tieng_viet`, đúng số dòng và thứ tự của file test, mã hóa `utf-8-sig` giống file mẫu. **Vòng này chỉ chấm `private_submission.csv`.**

In [ ]:
def make_submission(src_lines, name):
    hyps = translate_ids(final_models, [encode_src(s) for s in src_lines], final_beam, LENPEN)
    df = pd.DataFrame({"tieng_trung": src_lines, "tieng_viet": hyps})
    csv_path = os.path.join(OUT_DIR, f"{name}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    with zipfile.ZipFile(os.path.join(OUT_DIR, f"{name}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv_path, arcname=f"{name}.csv")

    # Kiểm tra file nộp
    check = pd.read_csv(csv_path, encoding="utf-8-sig", keep_default_na=False)
    assert list(check.columns) == ["tieng_trung", "tieng_viet"]
    assert len(check) == len(src_lines), f"{len(check)} != {len(src_lines)}"
    assert (check["tieng_viet"].str.strip() != "").all(), "có bản dịch rỗng"
    print(f"{csv_path}: {len(check)} dòng, OK")
    return df

private_df = make_submission(private_zh, "private_submission")
public_df = make_submission(public_zh, "public_submission")
private_df.head(10)